In [ ]:
!rm -rf PINN_hessian

Connect to Google Drive (so save the data there)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Connect to github - to get the code from there

In [ ]:
from getpass import getpass
token = getpass('GitHub token: ')
token = ...
! git clone --branch main https://{token}@github.com/marjankowskam/Comparing_Dynamics


GitHub token: ··········
Cloning into 'PINN_hessian'...
remote: Enumerating objects: 860, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 860 (delta 51), reused 78 (delta 28), pack-reused 738 (from 2)
Receiving objects: 100% (860/860), 49.79 MiB | 14.00 MiB/s, done.
Resolving deltas: 100% (512/512), done.


In [ ]:
%cd Comparing_Dynamics

/content/PINN_hessian


Install Weights and Biases - thats where you can see all your runs.

In [ ]:
# !pip install wandb
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: marcelina-marjankowska (marcelina-marjankowska-university-college-london-ucl-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!pip install -r requirements.txt
# import matplotlib as mpl
# mpl.rcParams.update(mpl.rcParamsDefault)
# !sudo apt-get install texlive-latex-extra texlive-fonts-recommended dvipng cm-super


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.0/154.0 kB 15.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 19.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 7.1 MB/s eta 0:00:00
ERROR: Ignored the following versions that require a different python version: 1.10.0 Requires-Python <3.12,>=3.8; 1.10.0rc1 Requires-Python <3.12,>=3.8; 1.10.0rc2 Requires-Python <3.12,>=3.8; 1.10.1 Requires-Python <3.12,>=3.8; 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.6.2 Requires-Python >=3.7,<3.10; 1.6.3 Requires-Pyt

## Training models and computing spectral information

In [ ]:
import torch
import torch.nn as nn
from tqdm import tqdm

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
import sys

sys.path.append('..')

from config import Config
from train import train_and_collect_metrics

import os
base_path = "/content/drive/MyDrive/STUDIA/PhD/Comparing_Dynamics/"

/content/PINN_hessian/src/train_utils.py:230: SyntaxWarning: invalid escape sequence '\ '
  match_number = re.compile('-?\ *[0-9]+\.?[0-9]*(?:[Ee]\ *-?\ *[0-9]+)?')
/content/PINN_hessian/tasks/pinn.py:436: SyntaxWarning: invalid escape sequence '\ '
  match_number = re.compile('-?\ *[0-9]+\.?[0-9]*(?:[Ee]\ *-?\ *[0-9]+)?')


In [ ]:
use_wandb = True
device = "cuda"

cfg = Config(
    initial_seed = 49,

    model_name="mnistMLP",
        model_params={
            "hidden_dim": [100, 100, 10],
            "activation_fn": "relu",
        },

    opt_name = "sgd",
    opt_params={
        "lr": 0.0001,
    },

    task_name = None,
    task_params = {
        "batch_size": None
    },

    experiment_name = "MNIST displacement - ablations",
    experiment_params = {
        "hessian_n_top": 3,
        "hessian_n_bottom": 0,
        "hessian_checkpoints": [],
        "hessian_eigv_tol": 0,
        "hessian_data_subset": "full",
        "num_epochs": None
    },

    num_epochs = None,
)

# Experiment: ablation studies on batch size
exp_opts = ["sgd"]
task_names = ["fashion_mnist"]
batch_sizes = [32, 64, 128, 1024, 60000]
seeds = [11, 19, 29]

print(f"computation expected to take {len(exp_opts)*len(task_names)*len(batch_sizes)*len(seeds)} hours")

for seed in seeds:
    cfg.initial_seed = seed
    for experiment in range(len(exp_opts)):
        cfg.opt_name = exp_opts[experiment]
        cfg.opt_params = {"lr": 0.001 } if exp_opts[experiment] == "adam" else {"lr": 0.01}
        for task_name in task_names:
            cfg.task_name = task_name

            for batch_size in batch_sizes:
                cfg.task_params = {"batch_size": batch_size}
                cfg.num_epochs = int(1e5 * 128/batch_size)
                cfg.experiment_params["num_epochs"] = cfg.num_epochs

                wb_run = wandb.init(
                    project="PINN_hessian_spectrum_v2",
                    notes="batch size ablation",
                    tags=["v3", "ablations"],
                    config = cfg,
                )

                result = train_and_collect_metrics(cfg, device, use_wandb)

                # save the hessian data
                hResults = result['hessian_object'].get_final_data()

                wandb.log({**result['test_metrics']})

                name = wb_run.name

                torch.save(hResults, base_path + f"hResults_{wb_run.name}.pt")
                torch.save(result['model_state_dict'], base_path + f'Model_{wb_run.name}.pt')

                cfg.save_to_file(wb_run.name, data_dir=base_path)

                wandb.finish()

computation expected to take 15 hours


fashion_mnist
{'batch_size': 32}


100%|██████████| 26.4M/26.4M [00:03<00:00, 8.00MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 180kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.30MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 8.33MB/s]
  0%|                                                                                             | 0/400000 [00:00<?, ?it/s]

loss weights updated: {'CrossEnt': 1.0}


/content/PINN_hessian/hessian.py:313: RuntimeWarning: divide by zero encountered in log10
  wandb.log({"log10 t": np.log10(step_id)}, step=step_id)
  0%|                                          | 99/400000 [00:05<1:37:32, 68.33it/s, CrossEnt 1.44e+00 / grad_norm 4.15e+00]

...

 81%|██████████████████████████████▋       | 323513/400000 [1:25:09<16:33, 77.02it/s, CrossEnt 3.43e-04 / grad_norm 5.41e-02]

loss weights updated: {'CrossEnt': 1.0}


 81%|██████████████████████████████▋       | 323609/400000 [1:25:11<18:00, 70.70it/s, CrossEnt 1.44e-03 / grad_norm 1.40e-01]

loss weights updated: {'CrossEnt': 1.0}


 81%|██████████████████████████████▋       | 323641/400000 [1:25:11<17:08, 74.22it/s, CrossEnt 6.34e-04 / grad_norm 1.40e-01]

In [ ]:
hResults

## Loading/ Saving the runs

In [ ]:
# base_path = "/content/drive/MyDrive/STUDIA/PhD/ICML_2026_Hessian_Eigenvectors/"

# import os

# name = wb_run.name

# # torch.save(result['update_norm'], base_path  + f'Update_{wb_run.name}.pt')

# torch.save(hResults, base_path + f"hResults_{wb_run.name}.pt")
# torch.save(result['model_state_dict'], base_path + f'Model_{wb_run.name}.pt')

# # torch.save(result['optimizer_state_dict'], base_path + f'Opt_{wb_run.name}.pt')

# cfg.save_to_file(wb_run.name, data_dir=base_path)

In [ ]:
# from google.colab import files

# if use_wandb:
#     name = wb_run.name
#     files.download(f'Config_{name}')
#     files.download(f'Model_{name}.pt')
#     # files.download(f'hResults_{name}.pt')
#     files.download(f'Opt_{name}.pt')
#     files.download(f'Update_{name}.pt')

#     # wandb.finish()

In [ ]:
#  files.download(f'hResults_{name}.pt')

## Plotting and Logging

In [ ]:
# def plot_tw_loglog(hResults, metric, title="", ylabel="", tw_min=0, tw_max = 100*100*100, normalize=False):
#   resultsMetric = hResults[metric]

#   data_t = {str(tw): [] for tw in hResults['tw_list']}
#   data_metric = {str(tw): [] for tw in hResults['tw_list']}

#   for i, (tw, t) in enumerate(hResults['indexes']):
#     data_t[str(tw)].append(t)
#     value = resultsMetric[i].cpu()
#     if normalize:
#         time_index, = np.where(hResults['all_timesteps'] == tw + t)
#         Dt = hResults['D_t'][time_index[0]]
#         value = resultsMetric[i].cpu()/Dt
#     data_metric[str(tw)].append(value)

#   fig, ax = plt.subplots()
#   cmap = plt.get_cmap('viridis')
#   colors = cmap(np.linspace(0, 1, len(hResults['tw_list'])))
#   color_map = {str(tw): color for tw, color in zip(hResults['tw_list'], colors)}

#   for tw in data_t:
#     if int(tw) >= tw_min and int(tw) <= tw_max and len(data_metric[tw]) > 0:
#         ax.loglog(data_t[tw], data_metric[tw], label=f"tw={tw}", color=color_map[tw])

#   ax.set_xlabel("t [steps]")
#   ax.set_ylabel(ylabel)
#   ax.set_title(title)

#   ax.legend(
#     loc="center left",
#     bbox_to_anchor=(1.02, 0.5)
#   )

#   fig.tight_layout()
#   return fig

from plotting_utils import plot_tw_loglog

In [ ]:
# num_eigv = cfg.experiment_params["hessian_n_top"]
# # PLOT the progression of eigenvectors
# times = np.unique(np.append(hResults["all_timesteps"], hResults["tw_list"]))
# times = times[times < cfg.num_epochs]
# fig, ax = plt.subplots()
# for k in range(num_eigv):
#   eigvInf = hResults['eigv'][-1][k]
#   eigvDot = [abs(torch.dot(eigvInf, eigv[k]).item()) for eigv in hResults['eigv']]
#   ax.loglog(times, eigvDot, label = f"eigv{num_eigv-k}")

# ax.set_xlabel("t")
# ax.set_ylabel("dot(eigvInf, eigv(t))")
# ax.set_title("convergence of the top eigenvector")
# ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
# fig.tight_layout()

# # PLOT the displacement curve
# fig_w = plot_tw_loglog(hResults, 'weight_displacement', "Mean square weight displacement", "delta(tw, tw + t)", plot_type="plot")

# for k in range(num_eigv):
#   fig_e = plot_tw_loglog(hResults, f'eigv{k+1}_displacement', f"Mean square eigv{k+1} displacement", f"delta_e{k+1}(tw, tw + t)", plot_type="plot")
#   fig_e_dot = plot_tw_loglog(hResults, f'eigv{k+1}_alignment', f"Eigv{k+1} dot alignment", "dot(tw, tw + t)", plot_type="plot")
#   fig_ew = plot_tw_loglog(hResults, f'eigv{k+1}_weight_correlation', f"Eigv{k+1} and normalized weight change alignment", f"|dot(eigv{k+1}(tw), normalized(w(tw + t)-w(t))|", plot_type="plot")
# #   fig_eg = plot_tw_loglog(hResults, f'eigv{k+1}_dot_grad_vec', f"Dot of eigv{k+1} and gradient", f"dot(e{k+1}, grad)")

#   if use_wandb:
#       wandb.log({f"eigv{k+1}_displacement": wandb.Image(fig_e)})
#       plt.close(fig_e)
#       wandb.log({f"eigv{k+1}_weight_correlation": wandb.Image(fig_ew)})
#       plt.close(fig_ew)
#       wandb.log({f"eigv{k+1}_dot_alignment": wandb.Image(fig_e_dot)})
#       plt.close(fig_e_dot)
#     #   wandb.log({f"eigv{k+1}_dot_grad": wandb.Image(fig_eg)})
#     #   plt.close(fig_eg)

# if use_wandb:
#   wandb.log({"top eigenvector convergence": wandb.Image(fig)}); plt.close(fig)
#   # classic:
#   wandb.log({"weight_displacement": wandb.Image(fig_w)})
#   wandb.finish()

# plt.close(fig_w)

Plotting test


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from src.models import get_model
from src.train_utils import get_task
from config import Config


sam_runs =[f"FashionMNIST - SGD - seed {seed}" for seed in ["11"]] ###### SAM

# run_names = ["FashionMNIST - SGD - seed 19", "FashionMNIST - Adam - seed 39 - lr1e-3"]

groups = {
    # "Adam": [],
    "SGD": sam_runs,
    # "SGD": [],
}

runs = {
    # "Adam": [],
    "SGD": ['jedi-jawa-129'],
    # "SGD": [],
}



import gc

def load_data(groups, runs, metric_name):
    results_metric = {group_name: [] for group_name in groups.keys()}
    times = {group_name: [] for group_name in groups.keys()}
    results_metric = {mName:{group_name: [] for group_name in groups.keys()} for mName in metric_name}

    configs = []
    for group_name, group_runs in runs.items():
        for name in group_runs:
            r = torch.load(f"hResults_{name}.pt", weights_only=False, map_location="cpu")
            for mName in metric_name:
                metric = r[mName]
                results_metric[mName][group_name].append(metric)
            all_timesteps = r["all_timesteps"]; tw_list = r["tw_list"]; indexes = r["indexes"]

            del r
            gc.collect()
            print("loaded.")

        print(group_name, "loaded")
        cfg = Config.load_from_file(name, '')
        configs.append(cfg)
        t = np.unique(np.append(all_timesteps, tw_list))
        t = t[t < cfg.num_epochs]
        times[group_name] = t
        print(f"{group_name} loaded!")

    return results_metric, all_timesteps, tw_list, indexes, times, configs

In [ ]:
results, all_timesteps, tw_list, indexes, times, configs = load_data(groups, runs, metric_name=["eig", "eigv1_dot_update", "eigv1_dot_grad_vec", "eigv2_dot_update", "eigv2_dot_grad_vec", "eigv3_dot_update", "eigv3_dot_grad_vec"]) #6-16min loading

In [ ]:
np.asarray(results[f"eigv{eig}_dot_grad_vec"]["SGD"], dtype=float)

In [ ]:
# step 1: calculate the aggregated metric.
alignment = {}
for group in results["eig"].keys():
    alignment_in_group = {}
    for eig in range(1, 4):
        means = np.asarray(results[f"eigv{eig}_dot_update"][group], dtype=float).mean(axis=1)
        alignment_in_group[f"v_{eig}"] = means
    alignment[group] = alignment_in_group
print(alignment)

galignment = {}
for group in results["eig"].keys():
    alignment_in_group = {}
    for eig in range(1, 4):
        means = np.asarray(results[f"eigv{eig}_dot_grad_vec"][group], dtype=float).mean(axis=1)
        alignment_in_group[f"v_{eig}"] = means
    galignment[group] = alignment_in_group
galignment

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_metric1_vs_metric2(metric1_dict, metric2_dict,
                            xlabel="Metric 1",
                            ylabel="Metric 2",
                            title="Metric 1 vs Metric 2 by Optimizer",
                            savepath=None):

    rows = []

    for opt in metric1_dict.keys():
        m1_dict = metric1_dict[opt]
        m2_dict = metric2_dict[opt]

        for eig_key in m1_dict.keys():  # v_1, v_2, v_3
            m1 = np.asarray(m1_dict[eig_key], dtype=float).flatten()
            m2 = np.asarray(m2_dict[eig_key], dtype=float).flatten()

            n = min(len(m1), len(m2))

            for i in range(n):
                rows.append({
                    "Optimizer:": opt,
                    "Eigenvector:": eig_key,
                    "m1": m1[i],
                    "m2": m2[i],
                })

    df = pd.DataFrame(rows)

    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        "font.size": 16,          # base
        "axes.titlesize": 20,
        "axes.labelsize": 18,
        "xtick.labelsize": 13,
        "ytick.labelsize": 13,
        "legend.fontsize": 16,
        "legend.title_fontsize": 15,
        "lines.linewidth": 1.5,
    })
    plt.figure(figsize=(6, 3.5))


    eig_label_map = {
        "v_1": r"$v^{(1)}$",
        "v_2": r"$v^{(2)}$",
        "v_3": r"$v^{(3)}$",
    }
    df["Eigenvector:"] = df["Eigenvector:"].map(eig_label_map)

    markers = {
    r"$v^{(1)}$": "o",
    r"$v^{(2)}$": "X",
    r"$v^{(3)}$": "^"
    }

    sns.scatterplot(
        data=df,
        x="m1",
        y="m2",
        hue="Optimizer:",
        style="Eigenvector:",
        markers=markers,
        alpha=0.7,
        s=100,
        edgecolor="white",
        linewidth=0.7
    )

    min_val = min(df["m1"].min(), df["m2"].min()) -0.02
    max_val = max(df["m1"].max(), df["m2"].max()) + 0.02

    plt.xlim(min_val, max_val)
    plt.ylim(min_val, max_val)

    # keep geometric equality
    plt.gca().set_aspect("equal", adjustable="box")

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(title="", frameon=False, bbox_to_anchor=(1.05, 1), ncols=2)
    plt.tight_layout()
    plt.show()

    if savepath:
        plt.savefig(savepath, bbox_inches="tight")

    plt.show()

plot_metric1_vs_metric2(alignment, galignment)

# To run

In [ ]:
# """Example: PINN"""
# cfg = Config(

#     task_name = "pinn",
#     task_params = {
#         "pde_name": 'convection',
#         "pde_params": ["beta", "40"],
#     },
#     #     pde_name="reaction",
# #     pde_params=["rho", "5"],

# #     pde_name = "wave",
# #     pde_params=["beta", "5"],

#     model_name = "pinn",
#     model_params = {
#         "hidden_dim": 200,
#         "num_layers": 4,
#     },

#     opt_name = "adam_lbfgs",
#     opt_params={
#         # "lr": 0.001,
# #         "switch_epochs": 500,
#         "adam_lr": 0.001,
#         "lbfgs_history_size": 100,
#         "switch_epoch_lbfgs": 5000,
#     },

#     experiment_name = "displacement",
#     experiment_params = {
#         "hessian_n_top": 3,
#         "hessian_n_bottom": 0,
#         "hessian_checkpoints": [],
#         "hessian_eigv_tol": 1e-3,
#     },

#     num_epochs = 10000,
# )

# Archive

In [ ]:

#     Config(
#     initial_seed=19,

#     pde_name= None,
#     pde_params= None,

#     opt_name="sgd",
#     opt_params={
#         "lr": 0.01
#     },

#     num_epochs=10,

#     hessian_experiment_params = {
#         "hessian_n_top": 6,
#         "hessian_n_bottom": 0,
#         "hessian_checkpoints": [0,200,400,550,600,800],
#     },
#     batch_size = 128

#     # model
#     hidden_dim = [100, 100, 10],
#     activation_fn = nn.ReLU,
# ),


In [ ]:
     # reaction
#     Config(
#     initial_seed=19,

#     pde_name= "convection",
#     pde_params= ["beta", "40"],

#     opt_name="adam_lbfgs_nncg",
#     opt_params={
#         "switch_epochs": 500,
#         "adam_lr": 0.001,
#         "lbfgs_history_size": 100,
#         "switch_epoch_lbfgs": 500,
#         "switch_epoch_nncg": 900,
#         "precond_update_freq": 1,
#         "nncg_mu": 0.001,
#         "nncg_rank": 60
#     },

#     num_epochs=1000,

#     hessian_experiment_params = {
#         "hessian_n_top": 6,
#         "hessian_n_bottom": 0,
#         "hessian_checkpoints": [0,200,400,550,600,800],
#     }
# ),
#     Config(
#     initial_seed=19,

#     pde_name="reaction",
#     pde_params=["rho", "5"],

#     opt_name="adam_lbfgs_nncg",
#     opt_params={
#         "switch_epochs": 500,
#         "adam_lr": 0.001,
#         "lbfgs_history_size": 100,
#         "switch_epoch_lbfgs": 500,
#         "switch_epoch_nncg": 900,
#         "precond_update_freq": 1,
#         "nncg_mu": 0.001,
#         "nncg_rank": 60
#     },

#     num_epochs=1000,

#     hessian_experiment_params = {
#         "hessian_n_top": 6,
#         "hessian_n_bottom": 0,
#         "hessian_checkpoints": [0,200,400,550,600,800],
#     }
# ),
#     Config(
#     initial_seed=19,

#     pde_name = "wave",
#     pde_params=["beta", "5"],

#     opt_name="adam_lbfgs_nncg",
#     opt_params={
#         "switch_epochs": 500,
#         "adam_lr": 0.001,
#         "lbfgs_history_size": 100,
#         "switch_epoch_lbfgs": 500,
#         "switch_epoch_nncg": 900,
#         "precond_update_freq": 1,
#         "nncg_mu": 0.001,
#         "nncg_rank": 60
#     },

#     num_epochs=1000,

#     hessian_experiment_params = {
#         "hessian_n_top": 6,
#         "hessian_n_bottom": 0,
#         "hessian_checkpoints": [0,200,400,550,600,800],
#     }
# )

In [ ]:

run_specs = [
#   {
#    'initial_seed': 19, # 345,
#    'pde_name': "convection",
#    'pde_params': ["beta", "40"],
#    'opt_name': "adam_lbfgs_nncg",
#    'switch_epochs': 500,
#    'adam_lr': 0.0001,
#    'lbfgs_history_size': 100,
#    'num_layers': 4,
#    'num_neurons': 200,
#    'loss_name': "mse",
#    'num_x': 257,
#    'num_t': 101,
#    'num_res': 10000,
#    'num_epochs': 1000, #12500,
#    'balancer_type': "Fixed",
#    'update_balancer_every_iter': 100
#   },
#   {
#    'initial_seed': 19,#456,
#    'pde_name': "reaction",
#    'pde_params': ["rho", "5"],

#    'opt_name': "adam_lbfgs_nncg",
#    'opt_params': ["switch_epochs", 500, "adam_lr", 0.001, "lbfgs_history_size",100, 'switch_epoch_lbfgs', 500, 'switch_epoch_nncg', 750, 'precond_update_freq', 20, 'nncg_mu', 0.001, 'nncg_rank', 60]

#    'num_layers': 4,
#    'num_neurons': 200,
#    'loss_name': "mse",
#    'num_x': 257,
#    'num_t': 101,
#    'num_res': 10000,

#    'num_epochs': 1000, #12500,

#    'balancer_type': "Fixed",
#    'update_balancer_every_iter': 100,
#   },
#   {
#   'initial_seed': 19, #567,
#    'pde_name': "wave",
#    'pde_params': ["beta", "5"],
#    'opt_name': "adam_lbfgs_nncg",
#    'switch_epochs': 500, #11000,
#    'adam_lr': 0.001,
#    'lbfgs_history_size': 100,
#    'num_layers': 4,
#    'num_neurons': 200,
#    'loss_name': "mse",
#    'num_x': 257,
#    'num_t': 101,
#    'num_res': 10000,
#    'num_epochs': 1000, #12500,
#     'balancer_type': "Fixed",
#    'update_balancer_every_iter': 100
#   }
]